# Human validated indicator categories 

In [1]:
from pathlib import Path
import re

import pandas as pd


# ============================================================
# Stage: User settings
# ============================================================

FIGURE_DIR = Path(".")

COUNTRY_AVAILABILITY_FILE = FIGURE_DIR / "combined_country_availability_final.csv"
HUMAN_VALIDATION_FILE = FIGURE_DIR / "LLM_indicator_coding_workflow - Human Validation_MC.xlsx"
HUMAN_VALIDATION_SHEET = "Step5_Human_Validation"

INDICATOR_CODING_FILE = FIGURE_DIR / "indicator_coding.xlsx"

COUNTRY_AVAILABILITY_WITH_CODING_CSV = (
    FIGURE_DIR / "combined_country_availability_final_with_indicator_coding.csv"
)

COUNTRY_AVAILABILITY_WITH_CODING_XLSX = (
    FIGURE_DIR / "combined_country_availability_final_with_indicator_coding.xlsx"
)


# ============================================================
# Stage: Coding columns
# ============================================================

DATA_CATEGORY_COLS = ["Stock", "Flow", "Policy_Guide", "Action"]
BOUNDARY_COLS = ["HPS", "DDP", "MS-NMS", "ADE"]

CODING_COLS = DATA_CATEGORY_COLS + BOUNDARY_COLS


# ============================================================
# Stage: Helper functions
# ============================================================

def clean_text(value):
    """
    Convert missing values to blank text and remove extra spaces.
    """
    if pd.isna(value):
        return ""
    return str(value).strip()


def is_human_flag(value):
    """
    Convert human validation entries into 1 or 0.

    Values treated as 1:
    1, A, Y, YES, TRUE, APPLICABLE, X
    """
    value = clean_text(value).upper()
    return value in {"1", "A", "Y", "YES", "TRUE", "APPLICABLE", "X"}


def parse_llm_boundary_flags(value):
    """
    Parse LLM boundary flags such as:
    HPS; DDP
    HPS, DDP
    HPS | DDP

    Returns a set of flags.
    """
    value = clean_text(value).upper()

    if value in {"", "0", "NONE", "N/A", "NA", "NULL"}:
        return set()

    return {
        item.strip()
        for item in re.split(r"[;,|]", value)
        if item.strip()
    }


def get_final_category(row):
    """
    Use the human-reviewed category first.
    If the human-reviewed category is blank, use the LLM category.

    Expected category codes:
    S = Stock
    F = Flow
    P = Policy Guide
    A = Action
    """
    human_category = clean_text(row["new_category_label"]).upper()
    llm_category = clean_text(row["llm_primary_category"]).upper()

    return human_category or llm_category


def get_boundary_flag(row, flag_name):
    """
    Use the human-reviewed boundary flag first.
    If the human-reviewed flag is blank, use the LLM boundary flags.
    """
    human_value = row[flag_name]

    if clean_text(human_value) != "":
        return 1 if is_human_flag(human_value) else 0

    llm_flags = parse_llm_boundary_flags(row["llm_boundary_flags"])

    return 1 if flag_name.upper() in llm_flags else 0


# ============================================================
# Stage 1: Prepare indicator coding
# Convert human validation into final binary codes
# Output: indicator_coding.xlsx
# ============================================================

df_validation = pd.read_excel(
    HUMAN_VALIDATION_FILE,
    sheet_name=HUMAN_VALIDATION_SHEET,
    header=1
)

base_cols = [
    "no",
    "indicator",
    "Definition",
    "unit",
    "source",
    "original",
    "dataset"
]

category_cols = [
    "new_category_label",
    "llm_primary_category"
]

boundary_source_cols = [
    "llm_boundary_flags",
    "HPS",
    "DDP",
    "MS-NMS",
    "ADE"
]

required_cols = base_cols + category_cols + boundary_source_cols

missing_cols = [
    col for col in required_cols
    if col not in df_validation.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns in the human validation sheet: {missing_cols}"
    )


# Create final category based on human validation first, then LLM coding.
final_category = df_validation.apply(get_final_category, axis=1)


# Create the indicator coding table.
indicator_coding = df_validation[base_cols].copy()

indicator_coding["Stock"] = (final_category == "S").astype(int)
indicator_coding["Flow"] = (final_category == "F").astype(int)
indicator_coding["Policy_Guide"] = (final_category == "P").astype(int)
indicator_coding["Action"] = (final_category == "A").astype(int)


# Add accounting boundary flags.
for flag_name in BOUNDARY_COLS:
    indicator_coding[flag_name] = df_validation.apply(
        lambda row: get_boundary_flag(row, flag_name),
        axis=1
    )


# Check duplicate indicator IDs before saving.
if indicator_coding["no"].duplicated().any():
    duplicated_nos = (
        indicator_coding.loc[indicator_coding["no"].duplicated(), "no"]
        .tolist()
    )

    raise ValueError(
        f"Duplicate values found in indicator coding column 'no': "
        f"{duplicated_nos[:20]}"
    )


# Save indicator coding.
indicator_coding.to_excel(
    INDICATOR_CODING_FILE,
    sheet_name="indicator_coding",
    index=False
)

print(f"Saved indicator coding file: {INDICATOR_CODING_FILE}")


# ============================================================
# Stage 2: Merge coding with country availability
# Add Stock, Flow, Action, Policy Guide, and boundary flags
# Output: combined_country_availability_final_with_indicator_coding.csv
# ============================================================

availability = pd.read_csv(COUNTRY_AVAILABILITY_FILE)

if "no" not in availability.columns:
    raise ValueError("The country availability file must contain a 'no' column.")

if availability["no"].duplicated().any():
    duplicated_nos = (
        availability.loc[availability["no"].duplicated(), "no"]
        .tolist()
    )

    raise ValueError(
        f"Duplicate values found in availability column 'no': "
        f"{duplicated_nos[:20]}"
    )


# Remove old coding columns if they already exist.
availability_clean = availability.drop(
    columns=[col for col in CODING_COLS if col in availability.columns],
    errors="ignore"
)


# Keep only the coding columns needed for merging.
indicator_coding_for_merge = indicator_coding[["no"] + CODING_COLS].copy()


# Merge by indicator ID.
merged = availability_clean.merge(
    indicator_coding_for_merge,
    on="no",
    how="left",
    validate="one_to_one"
)


# Check unmatched rows.
unmatched_mask = merged[CODING_COLS].isna().any(axis=1)

if unmatched_mask.any():
    missing_nos = merged.loc[unmatched_mask, "no"].tolist()[:20]

    raise ValueError(
        f"{int(unmatched_mask.sum())} rows did not match indicator coding. "
        f"Example 'no' values: {missing_nos}"
    )


# Convert coding columns to integer.
merged[CODING_COLS] = merged[CODING_COLS].astype(int)


# Move coding columns after the dataset column, if dataset exists.
cols = list(merged.columns)

for col in CODING_COLS:
    cols.remove(col)

if "dataset" in cols:
    dataset_position = cols.index("dataset")
    merged = merged[
        cols[:dataset_position + 1]
        + CODING_COLS
        + cols[dataset_position + 1:]
    ]


# Save merged outputs.
merged.to_csv(
    COUNTRY_AVAILABILITY_WITH_CODING_CSV,
    index=False,
    encoding="utf-8-sig"
)

merged.to_excel(
    COUNTRY_AVAILABILITY_WITH_CODING_XLSX,
    sheet_name="combined_with_coding",
    index=False
)

print(f"Saved merged CSV file: {COUNTRY_AVAILABILITY_WITH_CODING_CSV}")
print(f"Saved merged Excel file: {COUNTRY_AVAILABILITY_WITH_CODING_XLSX}")


# ============================================================
# Optional preview
# ============================================================

display(indicator_coding.head())
display(merged.head())

C:\Users\landa\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


Saved indicator coding file: indicator_coding.xlsx
Saved merged CSV file: combined_country_availability_final_with_indicator_coding.csv
Saved merged Excel file: combined_country_availability_final_with_indicator_coding.xlsx


,no,indicator,Definition,unit,source,original,dataset,Stock,Flow,Policy_Guide,Action,HPS,DDP,MS-NMS,ADE
0,1,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0001,wb,1,0,0,0,0,0,1,0
1,2,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0002,wb,1,0,0,0,0,0,1,0
2,3,Foreign assets | Total | Aggregation: per capi...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0003,wb,1,0,0,0,0,0,1,0
3,4,Foreign assets | Total | Aggregation: total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0004,wb,1,0,0,0,0,0,1,0
4,5,Foreign liabilities | Total | Aggregation: per...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0005,wb,1,0,0,0,0,0,1,0


,no,indicator,Definition,unit,source,original,dataset,Stock,Flow,Policy_Guide,...,Uruguay,Uzbekistan,Vanuatu,Venezuela,Viet Nam,Virgin Islands (U.S.),West Bank and Gaza,Yemen,Zambia,Zimbabwe
0,1,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0001,wb,1,0,0,...,1,0,0,0,1,0,0,0,1,1
1,2,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0002,wb,1,0,0,...,1,0,0,0,1,0,0,0,1,1
2,3,Foreign assets | Total | Aggregation: per capi...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0003,wb,1,0,0,...,1,0,0,0,1,0,0,0,1,1
3,4,Foreign assets | Total | Aggregation: total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0004,wb,1,0,0,...,1,0,0,0,1,0,0,0,1,1
4,5,Foreign liabilities | Total | Aggregation: per...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0005,wb,1,0,0,...,1,0,0,0,1,0,0,0,1,1


# Filter indicators for SEEA mapping

SEEA mapping does not include Action and Policy indicators, so we filter them out and retain only Stock and Flow indicators.


In [3]:
# ============================================================
# Filter out Policy Guide and Action indicators
# ============================================================

# Input file
input_file = Path("indicator_coding.xlsx")

# Output file
output_file = Path("indicator_coding_filtered_no_policy_action.xlsx")

# Read Excel file
df = pd.read_excel(input_file)

# Clean column names to avoid issues with extra spaces
df.columns = df.columns.str.strip()

# Filter out rows where Policy Guide == 1 or Action == 1
df_filtered = df[
    ~(
        (df["Policy_Guide"] == 1) |
        (df["Action"] == 1)
    )
].copy()

# Save filtered data
df_filtered.to_excel(output_file, index=False)

print("Original rows:", len(df))
print("Filtered rows:", len(df_filtered))
print("Deleted rows:", len(df) - len(df_filtered))
print("Saved file:", output_file)

Original rows: 1320
Filtered rows: 566
Deleted rows: 754
Saved file: indicator_coding_filtered_no_policy_action.xlsx


# Monetary-Value Indicators

Identify an filter for monetary-value indicators


In [5]:
from pathlib import Path
import re

import pandas as pd


# ============================================================
# Stage: User settings
# ============================================================

FIGURE_DIR = Path(".")

INPUT_FILE = FIGURE_DIR / "combined_country_availability_final_with_indicator_coding.csv"

OUTPUT_MONETARY_ONLY_CSV = (
    FIGURE_DIR / "combined_country_availability_monetary_only.csv"
)

OUTPUT_MONETARY_ONLY_XLSX = (
    FIGURE_DIR / "combined_country_availability_monetary_only.xlsx"
)


# ============================================================
# Stage: Monetary unit keyword pattern
# ============================================================

MONETARY_UNIT_PATTERN = re.compile(
    r"("
    r"\$|us\$|u\.s\.\s*dollars?|usd|us\s*dollars?|dollars?|"
    r"current\s+us\$|constant\s+us\$|international\s+\$|"
    r"ppp|lcu|local\s+currency|national\s+currency|domestic\s+currency|currency\s+units?|"
    r"eur|euros?|\u20ac|gbp|pounds?|sterling|\u00a3|"
    r"yen|\u00a5|yuan|renminbi|rmb|"
    r"rupees?|\u20b9|pesos?|reais|real|francs?|"
    r"rand|naira|\u20a6|dirhams?|riyals?|dinars?|"
    r"shekels?|\u20aa|won|\u20a9|baht|dong|\u20ab|rupiah|rubles?|roubles?|\u20bd|lira|\u20ba"
    r")",
    re.IGNORECASE,
)


# ============================================================
# Stage: Helper functions
# ============================================================

def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def is_monetary_unit(value):
    unit_text = clean_text(value)

    if not unit_text:
        return False

    return bool(MONETARY_UNIT_PATTERN.search(unit_text))


def matched_monetary_terms(value):
    unit_text = clean_text(value)

    if not unit_text:
        return ""

    matches = sorted(
        {
            match.group(0)
            for match in MONETARY_UNIT_PATTERN.finditer(unit_text)
        },
        key=str.lower
    )

    return "; ".join(matches)


# ============================================================
# Stage: Load file
# ============================================================

df = pd.read_csv(INPUT_FILE)

if "unit" not in df.columns:
    raise ValueError("The input file must contain a 'unit' column.")


# ============================================================
# Stage: Identify monetary-value indicators
# ============================================================

df["is_monetary_unit"] = df["unit"].apply(is_monetary_unit)

df["monetary_unit_terms_found"] = df["unit"].apply(
    matched_monetary_terms
)


# ============================================================
# Stage: Filter only monetary-value indicators
# ============================================================

monetary_only = df.loc[df["is_monetary_unit"] == True].copy()


# ============================================================
# Stage: Save new files
# ============================================================

monetary_only.to_csv(
    OUTPUT_MONETARY_ONLY_CSV,
    index=False,
    encoding="utf-8-sig"
)

monetary_only.to_excel(
    OUTPUT_MONETARY_ONLY_XLSX,
    sheet_name="monetary_only",
    index=False
)


# ============================================================
# Stage: Preview
# ============================================================

print(f"Total indicators in original file: {len(df)}")
print(f"Total monetary-value indicators: {len(monetary_only)}")

print(f"Saved CSV: {OUTPUT_MONETARY_ONLY_CSV}")
print(f"Saved Excel: {OUTPUT_MONETARY_ONLY_XLSX}")

display(monetary_only.head())

Total indicators in original file: 1320
Total monetary-value indicators: 181
Saved CSV: combined_country_availability_monetary_only.csv
Saved Excel: combined_country_availability_monetary_only.xlsx


,no,indicator,Definition,unit,source,original,dataset,Stock,Flow,Policy_Guide,...,Vanuatu,Venezuela,Viet Nam,Virgin Islands (U.S.),West Bank and Gaza,Yemen,Zambia,Zimbabwe,is_monetary_unit,monetary_unit_terms_found
0,1,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0001,wb,1,0,0,...,0,0,1,0,0,0,1,1,True,US dollars
1,2,Domestic comprehensive wealth index | Total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0002,wb,1,0,0,...,0,0,1,0,0,0,1,1,True,US dollars
2,3,Foreign assets | Total | Aggregation: per capi...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0003,wb,1,0,0,...,0,0,1,0,0,0,1,1,True,US dollars
3,4,Foreign assets | Total | Aggregation: total | ...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0004,wb,1,0,0,...,0,0,1,0,0,0,1,1,True,US dollars
4,5,Foreign liabilities | Total | Aggregation: per...,SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...,US dollars,The Changing Wealth of Nations (CWON),wb_0005,wb,1,0,0,...,0,0,1,0,0,0,1,1,True,US dollars


In [6]:
from pathlib import Path
import pandas as pd


# ============================================================
# Stage: User settings
# ============================================================

FIGURE_DIR = Path(".")

INPUT_FILE = FIGURE_DIR / "combined_country_availability_monetary_only.csv"

OUTPUT_SUMMARY_CSV = FIGURE_DIR / "monetary_only_summary_by_dataset.csv"
OUTPUT_SUMMARY_XLSX = FIGURE_DIR / "monetary_only_summary_by_dataset.xlsx"


# ============================================================
# Stage: Load data
# ============================================================

df = pd.read_csv(INPUT_FILE)


# ============================================================
# Stage: Define columns to summarize
# ============================================================

GROUP_COL = "dataset"

SUMMARY_COLS = [
    "Stock",
    "Flow",
    "Policy_Guide",
    "Action",
    "HPS",
    "DDP",
    "MS-NMS",
    "ADE"
]


# ============================================================
# Stage: Check required columns
# ============================================================

required_cols = [GROUP_COL] + SUMMARY_COLS

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")


# ============================================================
# Stage: Clean binary columns
# ============================================================

for col in SUMMARY_COLS:
    df[col] = (
        pd.to_numeric(df[col], errors="coerce")
        .fillna(0)
        .clip(lower=0, upper=1)
        .astype(int)
    )


# ============================================================
# Stage: Create summary table by dataset
# ============================================================

summary_rows = []

for dataset_name, group_df in df.groupby(GROUP_COL, dropna=False):
    total_indicators = len(group_df)

    row = {
        "dataset": dataset_name,
        "total_indicators_left": total_indicators
    }

    for col in SUMMARY_COLS:
        number = int(group_df[col].sum())

        percent = (
            number / total_indicators * 100
            if total_indicators > 0
            else 0
        )

        row[f"{col}_number"] = number
        row[f"{col}_percent"] = round(percent, 2)

    summary_rows.append(row)

summary_by_dataset = pd.DataFrame(summary_rows)


# ============================================================
# Stage: Add total row across all datasets
# ============================================================

total_indicators = len(df)

total_row = {
    "dataset": "Total",
    "total_indicators_left": total_indicators
}

for col in SUMMARY_COLS:
    number = int(df[col].sum())

    percent = (
        number / total_indicators * 100
        if total_indicators > 0
        else 0
    )

    total_row[f"{col}_number"] = number
    total_row[f"{col}_percent"] = round(percent, 2)

summary_by_dataset = pd.concat(
    [summary_by_dataset, pd.DataFrame([total_row])],
    ignore_index=True
)


# ============================================================
# Stage: Save outputs
# ============================================================

summary_by_dataset.to_csv(
    OUTPUT_SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig"
)

summary_by_dataset.to_excel(
    OUTPUT_SUMMARY_XLSX,
    sheet_name="summary_by_dataset",
    index=False
)


# ============================================================
# Stage: Preview
# ============================================================

print(f"Saved CSV: {OUTPUT_SUMMARY_CSV}")
print(f"Saved Excel: {OUTPUT_SUMMARY_XLSX}")

display(summary_by_dataset)

Saved CSV: monetary_only_summary_by_dataset.csv
Saved Excel: monetary_only_summary_by_dataset.xlsx


,dataset,total_indicators_left,Stock_number,Stock_percent,Flow_number,Flow_percent,Policy_Guide_number,Policy_Guide_percent,Action_number,Action_percent,HPS_number,HPS_percent,DDP_number,DDP_percent,MS-NMS_number,MS-NMS_percent,ADE_number,ADE_percent
0,imf,104,0,0.00,20,19.23,84,80.77,0,0.0,0,0.00,0,0.0,18,17.31,8,7.69
1,unsd,1,0,0.00,1,100.00,0,0.00,0,0.0,0,0.00,0,0.0,1,100.00,0,0.00
2,wb,76,76,100.00,0,0.00,0,0.00,0,0.0,4,5.26,0,0.0,69,90.79,5,6.58
3,Total,181,76,41.99,21,11.60,84,46.41,0,0.0,4,2.21,0,0.0,88,48.62,13,7.18
